In [ ]:
import xarray as xr
import cartopy as ctpy
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from datetime import datetime, timedelta
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.patheffects as pe
import matplotlib.lines as mlines
import metpy.calc as mpcalc
import scipy.stats as stat
import seaborn as sns
import nctoolkit as nc
import pandas as pd
import dask.array as da
import xcdat
import importlib
import glob
from numpy import *
from metpy.calc import heat_index
from metpy.units import units
from netCDF4 import Dataset
from matplotlib.colors import LinearSegmentedColormap, ListedColormap
import sys

import filepaths as filepaths
import model_preprocess as model_prep
import calculations as dr_g_calcs
importlib.reload(filepaths)
importlib.reload(model_prep)
importlib.reload(dr_g_calcs)

In [ ]:
yr_i=2015
yr_f=2024
var_choice='pr'
label_choice='pr_mm'
plot_var='Precipitation'

timeframe='future'
label_timeframe='Future'

location_strs=['South Florida']
location_abbrevs=['South FL']
so_fl_ll=[25.,-82.]
so_fl_ur=[27.,-80.]
varss=['pr']
locations=[so_fl_ll,so_fl_ur]
lat_arr=array(locations)[:,0]
lon_arr=array(locations)[:,1]

insert coupled highiresmip into dictionary and process

In [ ]:
coupled_model_data={}

In [ ]:
def highresmip_insert_into_dict(dict_name,model_name):
    filepath_4_data=filepaths.highresmip_coupled_filepath(model=model_name,varname=var_choice,masked=True,timeframe=timeframe,monthly_scale=True)
    # print(filepath_4_data)
    data=model_prep.highresmips_open_file(filepath_4_data,varname=var_choice,yr_i=yr_i,yr_f=yr_f)
    data_ac=dr_g_calcs.annual_climatology(data,label_choice=label_choice,lat_loc=lat_arr,lon_loc=lon_arr,region_loc=True,latlon_flag=True)
    dict_name[model_name]=data_ac
    print('created key in dictionary: ',model_name)

In [ ]:
#future is only cmcc,fgoals,and hiram
if (timeframe=='historical'):
    highresmip_insert_into_dict(coupled_model_data,model_name='fgoals')
    highresmip_insert_into_dict(coupled_model_data,model_name='cmcc')
    highresmip_insert_into_dict(coupled_model_data,model_name='ecmwf')
    highresmip_insert_into_dict(coupled_model_data,model_name='hiram')
else:
    highresmip_insert_into_dict(coupled_model_data,model_name='fgoals')
    highresmip_insert_into_dict(coupled_model_data,model_name='cmcc')
    highresmip_insert_into_dict(coupled_model_data,model_name='hiram')

insert mesaclip into dictionary and process

In [ ]:
mesaclip_model_data={}

In [ ]:
importlib.reload(model_prep)
for i,j in enumerate(arange(1,11,1)):
    print('ensemble 0',str(j))
    datapath=filepaths.mesaclip_ihesp_filepath('0'+str(j),varname=var_choice,timeframe=timeframe,monthly_scale=True)
    data=model_prep.mesaclip_ihesp_open_file(datapath,varname=var_choice,yr_i=yr_i,yr_f=yr_f,ens_number='0'+str(j),timeframe=timeframe,monthly_scale=True,masked=True)
    data_ac=dr_g_calcs.annual_climatology(data,label_choice=label_choice,lat_loc=lat_arr,lon_loc=lon_arr,region_loc=True,latlon_flag=True)
    mesaclip_model_data[str(j)]=data_ac

insert uncoupled highresmip into dictionary and process

In [ ]:
uncoupled_model_data={}

In [ ]:
def highresmip_insert_into_dict(dict_name,model_name):
    filepath_4_data=filepaths.highresmip_uncoupled_filepath(model=model_name,varname=var_choice,masked=True,timeframe=timeframe,monthly_scale=True)
    data=model_prep.highresmips_open_file(filepath_4_data,varname=var_choice,yr_i=yr_i,yr_f=yr_f)
    data_ac=dr_g_calcs.annual_climatology(data,label_choice=label_choice,lat_loc=lat_arr,lon_loc=lon_arr,region_loc=True,latlon_flag=True)
    dict_name[model_name]=data_ac
    print('created key in dictionary: ',model_name)

In [ ]:
#future is only cam,fgoals,mri,cmcc,and hiram
if (timeframe=='historical'):
    highresmip_insert_into_dict(uncoupled_model_data,model_name='fgoals')
    highresmip_insert_into_dict(uncoupled_model_data,model_name='cmcc')
    highresmip_insert_into_dict(uncoupled_model_data,model_name='cam-mpas')
    highresmip_insert_into_dict(uncoupled_model_data,model_name='ecmwf')
    highresmip_insert_into_dict(uncoupled_model_data,model_name='hiram')
    highresmip_insert_into_dict(uncoupled_model_data,model_name='mri')
else:
    highresmip_insert_into_dict(uncoupled_model_data,model_name='fgoals')
    highresmip_insert_into_dict(uncoupled_model_data,model_name='cmcc')
    highresmip_insert_into_dict(uncoupled_model_data,model_name='cam-mpas')
    highresmip_insert_into_dict(uncoupled_model_data,model_name='hiram')
    highresmip_insert_into_dict(uncoupled_model_data,model_name='mri')

insert downscaled data into dictionary and process

In [ ]:
downscaled_data={}

In [ ]:
model_names=['cesm2','cnrm','hadgem3-mm','hadgem3-ll','ipsl','miroc6','mpi']

data_names=['loca','bccaq','gddp']

In [ ]:
importlib.reload(model_prep)
for i,j in enumerate(data_names):
    print('creating dictionary for dataset: ',j)
    downscaled_data[j]={}
    for k,l in enumerate(model_names):
        try:
            if (j=='loca'):
                filepath_4_data=filepaths.loca_filepath(model=l,varname=var_choice,monthly_scale=True,timeframe=timeframe,highres=False)
                data=model_prep.loca_monthly_open_file(filepath_4_data,varname=var_choice,yr_i=yr_i,yr_f=yr_f)
                data_ac=dr_g_calcs.annual_climatology(data,label_choice=label_choice,lat_loc=lat_arr,lon_loc=lon_arr,region_loc=True,latlon_flag=True)
                downscaled_data[j][l]=data_ac
                print('created key in dictionary: ',l)
            elif (j=='bccaq'):
                filepath_4_data=filepaths.bccaq_filepath(model=l,varname=var_choice,masked=True)
                data=model_prep.bccaq_open_file(filepath_4_data,varname=var_choice,yr_i=yr_i,yr_f=yr_f,monthly_scale=True)
                data_ac=dr_g_calcs.annual_climatology(data,label_choice=label_choice,lat_loc=lat_arr,lon_loc=lon_arr,region_loc=True,latlon_flag=True)
                downscaled_data[j][l]=data_ac
                print('created key in dictionary: ',l)
            else:
                filepath_4_data=filepaths.gddp_filepath(model=l,varname=var_choice)
                print(filepath_4_data)
                data=model_prep.gddp_open_file(filepath_4_data,varname=var_choice,yr_i=yr_i,yr_f=yr_f,monthly_scale=True)
                data_ac=dr_g_calcs.annual_climatology(data,label_choice=label_choice,lat_loc=lat_arr,lon_loc=lon_arr,region_loc=True,latlon_flag=True)
                downscaled_data[j][l]=data_ac
                print('created key in dictionary: ',l)
        except (OSError, ValueError) as e:
            print('no files')
            pass            

process observational data into dictionary

In [ ]:
obs_dict={}
labels=['era5','prism']

for k,l in enumerate(labels):
    print('prepping data: ',l)
    if (l=='era5'):
        files=filepaths.era5_filepath(var_choice,temporal='daily')
        ds=model_prep.era5_open_file(files,var_choice,str(yr_i),str(yr_f))
        ds.coords['lon'].attrs={'axis':"X"}
        ds.coords['lat'].attrs={'axis':"Y"}
        ds.coords['time'].attrs={'axis':"T"}
        obs_dict[l]=dr_g_calcs.annual_climatology(ds,label_choice,lat_loc=lat_arr,lon_loc=lon_arr,region_loc=True,latlon_flag=True)
    else:
        files=filepaths.prism_filepath(timeframe=timeframe)
        ds=model_prep.prism_original(files,str(yr_i),str(yr_f),monthly_scale=True)
        ds.coords['lon'].attrs={'axis':"X"}
        ds.coords['lat'].attrs={'axis':"Y"}
        ds.coords['time'].attrs={'axis':"T"}
        obs_dict[l]=dr_g_calcs.annual_climatology(ds,label_choice,lat_loc=lat_arr,lon_loc=lon_arr,region_loc=True,latlon_flag=True)

consolidate dictionaries and truncate into 1-d for plotting 

In [ ]:
%%time
coupled_array=[]
uncoupled_array=[]

informal_names=['cam-mpas','cmcc','fgoals','ecmwf','hiram','mri']
model_name = ['CAM-MPAS-HR','CMCC-CM2-VHR4','FGOALS-f3-H','ECMWF-IFS-HR','HiRAM-SIT-HR','MRI-AGCM3-2-S']

for i,j in enumerate(list(coupled_model_data.keys())):
    coupled_array.append(nanmean(coupled_model_data[j][label_choice].values,axis=(1,2)))
coupled_array=array(coupled_array)

for i,j in enumerate(list(uncoupled_model_data.keys())):
    uncoupled_array.append(nanmean(uncoupled_model_data[j][label_choice].values,axis=(1,2)))
uncoupled_array=array(uncoupled_array)

loca_array=[]
bccaq_array=[]
gddp_array=[]

informal_names=['cesm2','cnrm','hadgem3-mm','hadgem3-ll','ipsl','miroc6','mpi']
model_name = ['CESM2-LENS','CNRM-CM6-1','HadGEM3-GC31-MM','HadGEM3-GC31-LL','IPSL-CM6A-LR','MIROC6','MPI-ESM1-2-HR']

try:
    for i,j in enumerate(list(downscaled_data['loca'].keys())):
        loca_array.append(nanmean(downscaled_data['loca'][j][label_choice].values,axis=(1,2)))
except KeyError:
    print('loca not in variable dictionary')
loca_array=array(loca_array)

for i,j in enumerate(list(downscaled_data['bccaq'].keys())):
    bccaq_array.append(nanmean(downscaled_data['bccaq'][j][label_choice].values,axis=(1,2)))
print('bccaq done')
bccaq_array=array(bccaq_array)

for i,j in enumerate(list(downscaled_data['gddp'].keys())):
    gddp_array.append(nanmean(downscaled_data['gddp'][j][label_choice].values,axis=(1,2)))
print('gddp done')
gddp_array=array(gddp_array)

mesaclip_array=[nanmean(mesaclip_model_data['1'][label_choice].values,axis=(1,2)),
            nanmean(mesaclip_model_data['2'][label_choice].values,axis=(1,2)),
            nanmean(mesaclip_model_data['3'][label_choice].values,axis=(1,2)),
            nanmean(mesaclip_model_data['4'][label_choice].values,axis=(1,2)),
            nanmean(mesaclip_model_data['5'][label_choice].values,axis=(1,2)),
            nanmean(mesaclip_model_data['6'][label_choice].values,axis=(1,2)),
            nanmean(mesaclip_model_data['7'][label_choice].values,axis=(1,2)),
            nanmean(mesaclip_model_data['8'][label_choice].values,axis=(1,2)),
            nanmean(mesaclip_model_data['9'][label_choice].values,axis=(1,2)),
            nanmean(mesaclip_model_data['10'][label_choice].values,axis=(1,2))]

print('mesaclip done')
mesaclip_array=array(mesaclip_array)

obs_array=[nanmean(obs_dict['era5'][label_choice].values,axis=(1,2))]
obs_array.append(nanmean(obs_dict['chc_ucsb'][label_choice].values,axis=(1,2)))
obs_array.append(nanmean(obs_dict['prism'][label_choice].values,axis=(1,2)))
obs_array=array(obs_array)

print('done')

In [ ]:
mpl.rcParams['font.family'] = "sans-serif"
mpl.rcParams['font.sans-serif'] = "Verdana" 
plt.rcParams['savefig.dpi'] = 300
import matplotlib.path as mpath
import matplotlib.patches as mpatches

month_name=['J','F','M','A','M','J','J','A','S','O','N','D']
months=arange(0,len(month_name))
step=1
ylim_plot=[0,10]
ylim_tick=arange(0,11,step)
y_tick_label=['0','','2','','4','','6','','8','','10']
y_label='\nDaily Precipitation ($mm/day$)'

fig, ax = plt.subplots(nrows=3,ncols=2,figsize=(16,18),layout='constrained') #width,height
axs=ax.flat

categories=[coupled_array,loca_array,uncoupled_array,bccaq_array,mesaclip_array,gddp_array]
titles=['\nHighresMIP Coupled','\nLOCA Downscaling','\nHighresMIP Uncoupled','\nBCCAQ Downscaling','\nMESACLIP (CESM-HR)','\nNASA-NEX-GDDP Downscaling']

colors=['#1f77b4','#ff7f0e','#2ca02c','#d62728','#9467bd','#8c564b']


for i,j in enumerate(categories):
    
    axs[i].plot(months,nanmean(obs_array,axis=0),label='_nolabel_',linewidth=3.0,color='black',zorder=5,linestyle='--')
    axs[i].fill_between(months,percentile(obs_array,10,axis=0),percentile(obs_array,90,axis=0),alpha=0.2, color='gray',
                    label='_nolabel_', edgecolor='none',zorder=2)

    #########################################
    axs[i].plot(months,nanmean(j,axis=0), color=colors[i], linewidth=2.0, label='ENS Mean', alpha=0.85,zorder=3)
    axs[i].fill_between(months,percentile(j,10,axis=0),percentile(j,90,axis=0),alpha=0.3,
                        color=colors[i],linewidth=1.2,linestyle='-',edgecolor=colors[i],
                    label='10th-90th percentile', zorder=1) 
    
    axs[i].set_title(titles[i],fontsize=18,fontweight='bold')
    if (i>3):
        axs[i].set_xlabel('\nMonth of Year',fontsize=16)
    axs[i].set_xticks(months,labels=month_name,fontsize=14)
    if (i%2==0):
        axs[i].set_ylabel(y_label,fontsize=16)
    axs[i].tick_params(axis='y',labelsize=14)
    axs[i].legend(loc='upper left',fontsize=14.,ncols=1,fancybox=False,edgecolor='grey')
    
    axs[i].set_axisbelow(True)
    axs[i].set_ylim(ylim_plot[0],ylim_plot[1])
    axs[i].set_yticks(ylim_tick,labels=y_tick_label)
    
    axs[i].grid(True, which='major', axis='both', 
            linestyle='--', linewidth=0.5, alpha=0.3, color='gray', zorder=0)
    
    axs[i].spines['top'].set_visible(False)
    axs[i].spines['right'].set_visible(False)
    axs[i].set_xlim(-0.5, 11.5)

black_line = mlines.Line2D([], [], color='black',linestyle='--',label='Verification Data Mean')
black_patch = mpatches.Patch(facecolor='grey',alpha=0.25,label='Verification Data 10th-90th percentile')

handles=[black_line,black_patch]
    
fig.legend(handles=handles,loc='upper center',fontsize=14,ncols=2,bbox_to_anchor=(0.5, .945),fancybox=False,edgecolor='grey')

fig.suptitle('\n'+label_timeframe+' '+plot_var+' Annual Cycle: '+str(yr_i)+'-'+str(yr_f)+'\nSouthern Florida Spatial Mean: '+f"25\u00B0N - 27\u00B0N, 82\u00B0W - 80\u00B0W\n\n",
             fontweight='bold',fontsize=18)
